# Exploration initiale de la base SQLite

## Objectif

Ce notebook documente une première exploration de la base SQLite brute située dans `data/raw/risk_monitor_dataset.sqlite`.

L'objectif est de comprendre la structure des données et d'identifier des points potentiels de qualité avant toute étape de nettoyage, de transformation ou de scoring.

Aucune correction n'est appliquée dans ce notebook. Les cellules ci-dessous servent uniquement à observer les tables, leurs colonnes, leurs valeurs manquantes, les doublons possibles, les timestamps et les relations apparentes entre tables.

## 1. Chargement des dépendances et connexion SQLite

Cette section charge les bibliothèques nécessaires et prépare la connexion à la base SQLite brute.

In [3]:
from pathlib import Path
import sqlite3
import pandas as pd

In [4]:
DB_PATH = Path("../data/raw/risk_monitor_dataset.sqlite")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Base SQLite introuvable : {DB_PATH}")

connection = sqlite3.connect(DB_PATH)
connection

## 2. Liste des tables

Cette section récupère les tables déclarées dans la base SQLite, sans hypothèse sur leur rôle métier.

In [5]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection,
)

tables

,name
0,complaints
1,memberships
2,payments
3,subscriptions
4,users


In [6]:
table_names = tables["name"].tolist()
table_names

['complaints', 'memberships', 'payments', 'subscriptions', 'users']

## 3. Aperçu de chaque table

Pour chaque table, on affiche le nombre de lignes, le schéma SQLite et les 5 premières lignes. Cette étape reste descriptive.

In [7]:
def quote_identifier(identifier: str) -> str:
    return '"' + identifier.replace('"', '""') + '"'


def inspect_table(table_name: str) -> None:
    quoted_table_name = quote_identifier(table_name)

    row_count = pd.read_sql_query(
        f"SELECT COUNT(*) AS row_count FROM {quoted_table_name};",
        connection,
    )
    schema = pd.read_sql_query(f"PRAGMA table_info({quoted_table_name});", connection)
    preview = pd.read_sql_query(f"SELECT * FROM {quoted_table_name} LIMIT 5;", connection)

    print("=" * 80)
    print(f"Table : {table_name}")
    display(row_count)
    display(schema[["name", "type", "notnull", "pk"]])
    display(preview)


for table_name in table_names:
    inspect_table(table_name)

Table : complaints


,row_count
0,1213


,name,type,notnull,pk
0,id,INTEGER,0,1
1,reporter_id,INTEGER,0,0
2,target_id,INTEGER,0,0
3,subscription_id,INTEGER,0,0
4,type,TEXT,0,0
5,status,TEXT,0,0
6,created_at,TEXT,0,0
7,resolved_at,TEXT,0,0
8,resolution,TEXT,0,0


,id,reporter_id,target_id,subscription_id,type,status,created_at,resolved_at,resolution
0,1,904,1565,389,billing_issue,open,2023-09-06 08:02:33,NaN,NaN
1,2,1702,600,260,subscription_inactive,escalated,2024-07-30 12:44:00,NaN,NaN
2,3,1361,672,124,Accès refusé,resolved,2022-10-01 21:19:13,2024-01-27 20:51:20,
3,4,19,1706,317,ACCESS_DENIED,RESOLVED,2024-05-08 04:05:41,2024-06-03 03:41:59,refunded
4,5,327,609,289,ACCESS_DENIED,closed,2021-06-11 13:01:44,2021-10-31 06:56:01,refunded


Table : memberships


...

,id,brand,owner_id,created_at,status,max_slots,price_cents,currency
0,1,Microsoft 365,1,2023-01-14 23:23:57,0,4,1070,eur
1,2,HBO Max,1670,2023-07-16 21:39:51,2,4,233,USD
2,3,ChatGPT Plus,558,2024-10-02 13:54:25,0,3,1367,EUR
3,4,Microsoft 365,567,2023-10-26 23:03:50,0,6,460,EUR
4,5,NordVPN,950,2024-11-30 08:35:40,0,4,464,EUR


Table : users


,row_count
0,2001


,name,type,notnull,pk
0,id,INTEGER,0,1
1,email,TEXT,0,0
2,country,TEXT,0,0
3,signup_date,TEXT,0,0
4,status,INTEGER,0,0
5,last_seen,TEXT,0,0
6,referral_code,TEXT,0,0
7,phone_prefix,TEXT,0,0


,id,email,country,signup_date,status,last_seen,referral_code,phone_prefix
0,1,user_1@yahoo.fr,IT,2021-02-09 21:26:47,1,2021-04-28 14:23:00,NaN,+49
1,2,user_2@gmail.com,CH,2022-01-09T06:49:44Z,0,2023-01-05 17:03:20,NaN,+41
2,3,user_3@outlook.com,BE,1584765297,1,2021-07-30 05:38:37,27cb14dc,+32
3,4,user_4@hotmail.com,DE,2020-07-13T23:53:58Z,0,2024-06-10 09:06:04,NaN,+49
4,5,user_5@outlook.com,DE,21/07/2020 08:54,0,2026-03-06 20:16:50,NaN,+49


## 4. Analyse des valeurs manquantes

Cette section mesure les valeurs manquantes par table et par colonne. Elle affiche le nombre et le pourcentage de valeurs manquantes, sans remplacer, supprimer ou corriger aucune valeur.

In [8]:
missing_summaries = {}

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)

    row_count = len(df)
    missing_summary = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": df.isna().sum().values,
            "missing_percentage": (df.isna().mean().values * 100).round(2),
        }
    ).sort_values("missing_count", ascending=False)
    missing_summary.insert(0, "table", table_name)
    missing_summary.insert(1, "row_count", row_count)
    missing_summaries[table_name] = missing_summary

    print("=" * 80)
    print(f"Table : {table_name}")
    display(missing_summary)

Table : complaints


,table,row_count,column,missing_count,missing_percentage
8,complaints,1213,resolution,765,63.07
7,complaints,1213,resolved_at,700,57.71
0,complaints,1213,id,0,0.00
1,complaints,1213,reporter_id,0,0.00
2,complaints,1213,target_id,0,0.00
4,complaints,1213,type,0,0.00
3,complaints,1213,subscription_id,0,0.00
6,complaints,1213,created_at,0,0.00
5,complaints,1213,status,0,0.00


Table : memberships


,table,row_count,column,missing_count,missing_percentage
6,memberships,1083,reason,706,65.19
5,memberships,1083,left_at,637,58.82
0,memberships,1083,id,0,0.00
2,memberships,1083,subscription_id,0,0.00
1,memberships,1083,user_id,0,0.00
4,memberships,1083,joined_at,0,0.00
3,memberships,1083,status,0,0.00


Table : payments


,table,row_count,column,missing_count,missing_percentage
9,payments,7277,stripe_error_code,5444,74.81
7,payments,7277,captured_at,3370,46.31
1,payments,7277,user_id,0,0.00
0,payments,7277,id,0,0.00
2,payments,7277,subscription_id,0,0.00
3,payments,7277,amount_cents,0,0.00
5,payments,7277,status,0,0.00
4,payments,7277,fee_cents,0,0.00
6,payments,7277,created_at,0,0.00
8,payments,7277,currency,0,0.00


Table : subscriptions


,table,row_count,column,missing_count,missing_percentage
0,subscriptions,400,id,0,0.0
1,subscriptions,400,brand,0,0.0
2,subscriptions,400,owner_id,0,0.0
3,subscriptions,400,created_at,0,0.0
4,subscriptions,400,status,0,0.0
5,subscriptions,400,max_slots,0,0.0
6,subscriptions,400,price_cents,0,0.0
7,subscriptions,400,currency,0,0.0


Table : users


,table,row_count,column,missing_count,missing_percentage
6,users,2001,referral_code,1418,70.86
7,users,2001,phone_prefix,419,20.94
4,users,2001,status,25,1.25
0,users,2001,id,0,0.00
3,users,2001,signup_date,0,0.00
2,users,2001,country,0,0.00
1,users,2001,email,0,0.00
5,users,2001,last_seen,0,0.00


### Commentaires à compléter manuellement

- Observations sur les colonnes avec le plus de valeurs manquantes :
- Questions à clarifier avant nettoyage :
- Décisions possibles à reporter dans la section dédiée :

## 5. Détection des doublons exacts

Cette section compte les lignes exactement dupliquées dans chaque table. Elle ne déduit pas encore si ces doublons sont problématiques et ne supprime aucune ligne.

In [9]:
exact_duplicate_summaries = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    duplicate_count = int(df.duplicated().sum())
    row_count = len(df)

    exact_duplicate_summaries.append(
        {
            "table": table_name,
            "row_count": row_count,
            "exact_duplicate_rows": duplicate_count,
            "exact_duplicate_percentage": round((duplicate_count / row_count) * 100, 2) if row_count else 0,
            "has_exact_duplicates": duplicate_count > 0,
        }
    )

exact_duplicate_summary = pd.DataFrame(exact_duplicate_summaries)
exact_duplicate_summary

,table,row_count,exact_duplicate_rows,exact_duplicate_percentage,has_exact_duplicates
0,complaints,1213,0,0.0,False
1,memberships,1083,0,0.0,False
2,payments,7277,0,0.0,False
3,subscriptions,400,0,0.0,False
4,users,2001,0,0.0,False


### Commentaires à compléter manuellement

- Tables concernées par des doublons exacts :
- Hypothèses à vérifier avant toute suppression :
- Décisions possibles à reporter dans la section dédiée :

## 6. Premiers doublons potentiels

Cette section propose une détection simple de candidats à examiner lorsque des colonnes de type identifiant existent. Elle cherche des valeurs répétées dans les colonnes nommées `id` ou se terminant par `_id`, sans conclure qu'il s'agit d'erreurs et sans supprimer de données.

In [10]:
potential_duplicate_summaries = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    id_columns = [
        column
        for column in df.columns
        if column.lower() == "id" or column.lower().endswith("_id")
    ]

    for column in id_columns:
        non_missing_values = df[column].dropna()
        value_counts = non_missing_values.value_counts()
        duplicated_values = value_counts[value_counts > 1]

        potential_duplicate_summaries.append(
            {
                "table": table_name,
                "id_column": column,
                "non_missing_values": int(non_missing_values.shape[0]),
                "duplicate_id_values": int(duplicated_values.shape[0]),
                "rows_with_duplicate_id": int(non_missing_values.isin(duplicated_values.index).sum()),
            }
        )

potential_duplicate_summary = pd.DataFrame(potential_duplicate_summaries)
potential_duplicate_summary.sort_values(
    ["rows_with_duplicate_id", "duplicate_id_values"],
    ascending=False,
) if not potential_duplicate_summary.empty else potential_duplicate_summary

,table,id_column,non_missing_values,duplicate_id_values,rows_with_duplicate_id
9,payments,subscription_id,7277,344,7259
8,payments,user_id,7277,745,7205
3,complaints,subscription_id,1213,264,1163
6,memberships,subscription_id,1083,277,1005
1,complaints,reporter_id,1213,346,941
2,complaints,target_id,1213,244,560
5,memberships,user_id,1083,216,469
11,subscriptions,owner_id,400,36,77
4,memberships,id,1083,18,37
0,complaints,id,1213,0,0


In [11]:
MAX_DUPLICATE_ID_VALUES_TO_DISPLAY = 20

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    id_columns = [
        column
        for column in df.columns
        if column.lower() == "id" or column.lower().endswith("_id")
    ]

    for column in id_columns:
        duplicated_values = df[column].dropna().value_counts()
        duplicated_values = duplicated_values[duplicated_values > 1]

        if duplicated_values.empty:
            continue

        print("=" * 80)
        print(f"Table : {table_name} | Colonne candidate : {column}")
        display(duplicated_values.head(MAX_DUPLICATE_ID_VALUES_TO_DISPLAY).to_frame("count"))

Table : complaints | Colonne candidate : reporter_id


,count
reporter_id,
454,8
1917,7
1399,7
793,7
529,6
1073,6
1272,6
284,6
1639,6


Table : complaints | Colonne candidate : target_id


,count
target_id,
1998,6
212,5
600,4
54,4
439,4
446,4
323,4
1687,4
512,4


Table : complaints | Colonne candidate : subscription_id


...

,count
user_id,
793,41
1728,37
1973,33
1896,31
1653,31
93,30
1032,30
1702,30
704,29


Table : payments | Colonne candidate : subscription_id


,count
subscription_id,
48,69
362,69
258,66
170,64
20,63
278,60
354,60
64,59
99,55


Table : subscriptions | Colonne candidate : owner_id


,count
owner_id,
1055,3
1663,3
1776,3
481,3
1901,3
567,2
836,2
317,2
101,2


### Commentaires à compléter manuellement

- Colonnes d'identifiant à examiner :
- Valeurs candidates à vérifier :
- Vérifications complémentaires nécessaires avant décision :

## 7. Valeurs catégorielles incohérentes

Cette section aide à repérer les colonnes textuelles avec des modalités proches, rares ou inattendues. Aucun regroupement ou remplacement n'est effectué.

In [12]:
MAX_UNIQUE_VALUES_TO_DISPLAY = 30

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    text_columns = df.select_dtypes(include="object").columns

    print("=" * 80)
    print(f"Table : {table_name}")

    if len(text_columns) == 0:
        print("Aucune colonne textuelle détectée par pandas.")
        continue

    for column in text_columns:
        value_counts = df[column].value_counts(dropna=False).head(MAX_UNIQUE_VALUES_TO_DISPLAY)
        print(f"\nColonne : {column}")
        display(value_counts.to_frame("count"))

Table : complaints

Colonne : type


C:\Users\COLOMBE KEDJA\AppData\Local\Temp\ipykernel_11816\2186462689.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


,count
type,
other,153
wrong_credentials,145
subscription_inactive,143
owner_unresponsive,142
ACCESS_DENIED,135
access_denied,134
billing_issue,127
fraud_suspicion,118
Accès refusé,116



Colonne : status


,count
status,
in_progress,194
open,192
RESOLVED,171
resolved,169
escalated,165
Open,164
closed,158


...

,count
last_seen,
2021-04-28 14:23:00,1
2023-01-05 17:03:20,1
2021-07-30 05:38:37,1
2024-06-10 09:06:04,1
2026-03-06 20:16:50,1
2024-01-29 05:53:30,1
2025-05-14 23:06:23,1
2023-09-29 23:22:32,1
2025-12-23 04:44:01,1



Colonne : referral_code


,count
referral_code,
NaN,1418
df55f8c3,3
5d2188f1,3
834d2f8c,3
27cb14dc,2
8f42841e,2
17c3a6b5,2
2cfca0d1,2
3bd8d820,2



Colonne : phone_prefix


,count
phone_prefix,
NaN,419
+34,205
+43,197
+39,188
+351,180
+33,170
+49,164
+32,164
+41,158


## 8. Problèmes de timestamps

Cette section cible les colonnes dont le nom suggère une date ou un timestamp. Les conversions sont exploratoires et servent uniquement à repérer des valeurs non interprétables par pandas.

In [ ]:
TIMESTAMP_KEYWORDS = ["date", "time", "timestamp", "created", "updated"]

timestamp_summaries = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)

    candidate_columns = [
        column
        for column in df.columns
        if any(keyword in column.lower() for keyword in TIMESTAMP_KEYWORDS)
    ]

    for column in candidate_columns:
        parsed = pd.to_datetime(df[column], errors="coerce", utc=True)
        timestamp_summaries.append(
            {
                "table": table_name,
                "column": column,
                "non_missing_values": int(df[column].notna().sum()),
                "unparsed_values": int((df[column].notna() & parsed.isna()).sum()),
                "min_parsed": parsed.min(),
                "max_parsed": parsed.max(),
            }
        )

pd.DataFrame(timestamp_summaries)

ValueError: Mixed timezones detected. Pass utc=True in to_datetime or tz='UTC' in DatetimeIndex to convert to a common timezone.

## 9. Analyse des champs `status` non documentés

Cette section recense les colonnes dont le nom contient `status` et affiche leurs distributions observées. Aucune signification métier n'est attribuée aux valeurs.

In [ ]:
for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    df = pd.read_sql_query(f"SELECT * FROM {quoted_table_name};", connection)
    status_columns = [column for column in df.columns if "status" in column.lower()]

    if not status_columns:
        continue

    print("=" * 80)
    print(f"Table : {table_name}")

    for column in status_columns:
        print(f"\nChamp status : {column}")
        display(df[column].value_counts(dropna=False).to_frame("count"))

## 10. Vérification de cohérence entre les tables

Cette section prépare des contrôles simples sur les clés potentielles et les colonnes communes. Elle ne suppose pas encore de relations métier ou de contraintes de clé étrangère.

In [ ]:
schemas = {}

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    schema = pd.read_sql_query(f"PRAGMA table_info({quoted_table_name});", connection)
    schemas[table_name] = schema

column_locations = []
for table_name, schema in schemas.items():
    for column in schema["name"]:
        column_locations.append({"column": column, "table": table_name})

column_locations = pd.DataFrame(column_locations)

common_columns = (
    column_locations.groupby("column")["table"]
    .apply(list)
    .reset_index(name="tables")
)
common_columns["table_count"] = common_columns["tables"].str.len()
common_columns = common_columns.sort_values(["table_count", "column"], ascending=[False, True])

common_columns[common_columns["table_count"] > 1]

In [ ]:
foreign_keys = []

for table_name in table_names:
    quoted_table_name = quote_identifier(table_name)
    table_foreign_keys = pd.read_sql_query(f"PRAGMA foreign_key_list({quoted_table_name});", connection)
    if not table_foreign_keys.empty:
        table_foreign_keys.insert(0, "source_table", table_name)
        foreign_keys.append(table_foreign_keys)

if foreign_keys:
    pd.concat(foreign_keys, ignore_index=True)
else:
    pd.DataFrame(columns=["source_table", "id", "seq", "table", "from", "to", "on_update", "on_delete", "match"])

## 11. Décisions de nettoyage

Cette section est volontairement laissée vide à ce stade.

Elle sera complétée après revue des observations ci-dessus, avec des décisions explicites et justifiées.

In [ ]:
# À compléter plus tard.
# Ne pas appliquer de nettoyage dans cette version du notebook.

## Fermeture de la connexion

Fermer la connexion SQLite une fois l'exploration terminée.

In [ ]:
connection.close()